In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import scipy
import xskillscore as xs
import numpy as np
import dask
from dask.distributed import Client

import albedo_functions as af

In [ ]:
def p_cal(ctrl, sens):
    a = ctrl.resample(time='YE').mean().var(dim='time')
    b = sens.resample(time='YE').mean().var(dim='time')

# Add a small constant to avoid division by zero
    epsilon = 1e-8
    a += epsilon
    b += epsilon

    F = a / b
    #df = len(ctrl.resample(time='YE').mean())
    df = ctrl.resample(time='YE').mean().sizes['time']
    p_value = scipy.stats.f.cdf(F, df, df)
    p_value = xr.Dataset({
        'p': xr.DataArray(p_value, dims=('lat', 'lon'),
                          coords={'lon': a['lon'],'lat': a['lat']})
    })
    return p_value

In [ ]:
def interannual_std(dset, n_bootstrap):
    """
    Calculate interannual standard deviation and its significance via bootstrapping.
    
    Parameters:
        dset (xarray.Dataset): Dataset containing 'time' dimension.
        varname (str): Variable name inside the dataset.
        n_bootstrap (int): Number of bootstrap iterations.
    
    Returns:
        xr.DataArray: observed_std
    """
    # Step 1: Group by year
    annual_mean = dset.groupby('time.year').mean('time')
    
    # Step 2: Observed std dev
    observed_std = annual_mean.std('year')
    
    # Step 3: Bootstrap resampling using xskillscore
    resampled = xs.resampling.resample_iterations_idx(
        annual_mean, n_bootstrap, dim='year', replace=True, dim_max=annual_mean.year.size
    )
    boot_std = resampled.std(dim='year')
    return observed_std

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
BASE_PATH = f'{POST_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
# Open datasets with Dask chunking
cvh_ctrl = xr.open_dataset(BASE_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc', chunks={'time': 12})['cvh']
cvl_ctrl = xr.open_dataset(BASE_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc', chunks={'time': 12})['cvl']
cvh_sens = xr.open_dataset(BASE_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc', chunks={'time': 12})['cvh']
cvl_sens = xr.open_dataset(BASE_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc', chunks={'time': 12})['cvl']
# Filter: remove low standard deviation data
cvh_ctrl = xr.where(cvh_ctrl.std('time') >= 0.00005, cvh_ctrl, np.nan)
cvh_sens = xr.where(cvh_sens.std('time') >= 0.00005, cvh_sens, np.nan)
cvl_ctrl = xr.where(cvl_ctrl.std('time') >= 0.00005, cvl_ctrl, np.nan)
cvl_sens = xr.where(cvl_sens.std('time') >= 0.00005, cvl_sens, np.nan)

In [ ]:
# Align times
cvh_ctrl = cvh_ctrl.sel(time=slice(cvh_sens.time[0], cvh_sens.time[-1]))
cvl_ctrl = cvl_ctrl.sel(time=slice(cvh_sens.time[0], cvh_sens.time[-1]))

In [ ]:
# Compute interannual stds
interannual_cvh_ctrl_std = af.land_seas_mask(interannual_std(cvh_ctrl, 1000))
interannual_cvh_sens_std = af.land_seas_mask(interannual_std(cvh_sens, 1000))
interannual_cvl_ctrl_std = af.land_seas_mask(interannual_std(cvl_ctrl, 1000))
interannual_cvl_sens_std = af.land_seas_mask(interannual_std(cvl_sens, 1000))

In [ ]:
# Compute p-values
p_value_cvh = af.land_seas_mask(p_cal(cvh_ctrl, cvh_sens))
p_value_cvl = af.land_seas_mask(p_cal(cvl_ctrl, cvl_sens))

In [ ]:
# Convert to Dataset and save
interannual_cvh_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_cvh_std.nc', compute=True)
interannual_cvh_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_cvh_std.nc', compute=True)
interannual_cvl_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_cvl_std.nc', compute=True)
interannual_cvl_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_cvl_std.nc', compute=True)
p_value_cvh.to_netcdf(f'{SAVE_PATH}delta_cvh_std_p.nc', compute=True)
p_value_cvl.to_netcdf(f'{SAVE_PATH}delta_cvl_std_p.nc', compute=True)